# Camada Bronze — Ingestão Bruta
**Objetivo:** Ler os arquivos CSV do Volume e a cotação do dólar via API do Banco Central,
criando tabelas Delta na camada Bronze com timestamp de ingestão.

In [0]:
# Parâmetros de data para a coleta da cotação. Podem ser sobrescritos via Jobs.
from datetime import datetime

data_inicio = '01-01-2016'
data_fim = datetime.today().strftime('%m-%d-%Y')

print(f'Período de cotação: {data_inicio} → {data_fim}')

In [0]:
catalogo = 'olist_catalog'
bronze_db_name = 'bronze'
landing_volume = 'olist_vol'
volume_path = f'/Volumes/{catalogo}/{bronze_db_name}/{landing_volume}'

In [0]:
# Criação do catálogo, schema e volume Bronze (idempotente).
spark.sql(f'CREATE CATALOG IF NOT EXISTS {catalogo}')
spark.sql(f'USE CATALOG {catalogo}')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {bronze_db_name}')
spark.sql(f'USE SCHEMA {bronze_db_name}')

spark.sql(f'CREATE VOLUME IF NOT EXISTS {landing_volume}')

In [0]:
# Lê um CSV do Volume e grava como tabela Delta na camada Bronze.
# Cada tabela recebe timestamp_ingestion para rastreabilidade.
# overwriteSchema=true tolera mudanças de esquema entre execuções;
# multiLine=true permite campos de texto com quebras de linha.
from pyspark.sql import functions as F

def ingest_csv_to_bronze(csv_filename: str, table_name: str) -> None:
    '''Lê um CSV do Volume e grava como tabela Delta na camada Bronze.'''
    path = f'{volume_path}/{csv_filename}'
    print(f'Ingerindo {path} → bronze.{table_name}')

    df = (
        spark.read
             .option('header', 'true')
             .option('inferSchema', 'true')
             .option('multiLine', 'true')          # tolera quebras de linha dentro de campos
             .option('escape', '"')                # trata aspas escapadas corretamente
             .csv(path)
             .withColumn('timestamp_ingestion', F.current_timestamp())  # marca temporal de ingestão
    )

    (
        df.write
          .format('delta')
          .mode('append')
          .option('overwriteSchema', 'true')   # tolera evolução de esquema
          .saveAsTable(f'bronze.{table_name}')
    )
    print(f'{df.count()} linhas carregadas.')

In [0]:
# Ingestão dos arquivos CSV do dataset Olist.
csv_table_map = [
    ('olist_customers_dataset.csv',              'tb_customers'),
    ('olist_geolocation_dataset.csv',            'tb_geolocalizacao'),
    ('olist_order_items_dataset.csv',            'tb_order_items'),
    ('olist_order_payments_dataset.csv',         'tb_order_payments'),
    ('olist_order_reviews_dataset.csv',          'tb_order_reviews'),
    ('olist_orders_dataset.csv',                 'tb_orders'),
    ('olist_products_dataset.csv',               'tb_products'),
    ('olist_sellers_dataset.csv',                'tb_sellers'),
    ('product_category_name_translation.csv',    'tb_product_category_name_translation'),
]

for csv_file, table in csv_table_map:
    ingest_csv_to_bronze(csv_file, table)

In [0]:
# Coleta da cotação do dólar (BCB/PTAX) via endpoint público do Banco Central.
# O período é controlado pelos parâmetros data_inicio/data_fim definidos no início do notebook.
import requests
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

url = (
    'https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/'
    'CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)'
    f'?@dataInicial=\'{data_inicio}\'&@dataFinalCotacao=\'{data_fim}\''
    '&$select=dataHoraCotacao,cotacaoCompra&$format=json'
)

print(f'Chamando API BCB:\n{url}')
response = requests.get(url, timeout=60)
response.raise_for_status()

registros = response.json().get('value', [])
print(f'Registros recebidos: {len(registros)}')

In [0]:
# Converte a resposta JSON em DataFrame Spark, padroniza tipos e grava em bronze.tb_cotacao_dolar.
# Lança exceção se a API retornar lista vazia (período inválido ou fora do ar).
if registros:
    df_cotacao = (
        spark.createDataFrame(registros)          # cria a partir da lista de dicts
             .withColumnRenamed('dataHoraCotacao', 'data_hora_cotacao')
             .withColumnRenamed('cotacaoCompra',   'cotacao_compra')
             .withColumn('cotacao_compra', F.col('cotacao_compra').cast('double'))
             .withColumn('data_hora_cotacao', F.to_timestamp('data_hora_cotacao'))
             .withColumn('timestamp_ingestion', F.current_timestamp())
    )

    (
        df_cotacao.write
                  .format('delta')
                  .mode('append')
                  .option('overwriteSchema', 'true')
                  .saveAsTable('bronze.tb_cotacao_dolar')
    )
    print(f'bronze.tb_cotacao_dolar gravada com {df_cotacao.count()} registros.')
else:
    raise ValueError('API do Banco Central retornou lista vazia. Verifique o período informado.')